# 07_04 — Feature Selection + Hyperparameter Tuning — Early-Stage

**Projeto:** `dsc_legal_intelligence`  
**Objetivo:** selecionar features, testar ablações por blocos e tunar hiperparâmetros para o modelo early-stage de risco de perda.

## Semântica do target

```text
target = 0 -> banco ganhou
target = 1 -> banco perdeu
```

A classe positiva deste notebook é **perda do banco** (`POS_LABEL = 1`).

## Papel deste notebook

Este notebook deve ser executado **antes da calibração**.  
A calibração, Venn-Abers, escolha de threshold final e faixas de risco ficam para o próximo notebook:

```text
07_05_calibracao_threshold_conformal.ipynb
```

## Entradas esperadas

Preferencialmente, geradas pelos notebooks anteriores:

```text
data/processed/abt_early_stage_v1_dev_hist.parquet
data/processed/abt_early_stage_v1_holdout_hist.parquet
data/processed/feature_list_early_stage_v1_hist.txt
```

Caso esses arquivos não existam, o notebook tenta usar os arquivos sem histórico:

```text
data/processed/abt_early_stage_v1_dev.parquet
data/processed/abt_early_stage_v1_holdout.parquet
data/processed/feature_list_early_stage_v1.txt
```

## Saídas principais

```text
outputs/reports/modelagem_early_stage/90_04_feature_inventory.csv
outputs/reports/modelagem_early_stage/91_04_block_ablation_summary.csv
outputs/reports/modelagem_early_stage/92_04_rf_tuning_results.csv
outputs/reports/modelagem_early_stage/93_04_holdout_best_tuned_metrics.csv
outputs/reports/modelagem_early_stage/94_04_holdout_topk_risco_perda.csv
outputs/reports/modelagem_early_stage/95_04_holdout_topk_prioridade_financeira.csv
outputs/reports/modelagem_early_stage/96_04_permutation_importance_holdout.csv
outputs/reports/modelagem_early_stage/98_04_summary_step_04.csv
models/modelagem_early_stage/best_model_07_04.joblib
```


In [ ]:
# ============================================================
# 1. Imports e configuração
# ============================================================

import os
import re
import json
import math
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn import set_config
set_config(transform_output="default")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    make_scorer,
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.inspection import permutation_importance
from scipy.stats import randint
import joblib

RANDOM_STATE = 42
POS_LABEL = 1  # 1 = banco perdeu
NEG_LABEL = 0  # 0 = banco ganhou

TARGET_COL = "target_banco_ganhou"  # nome herdado; semântica atual: 0 ganhou, 1 perdeu
DATE_COL = "Datainicial"
VALUE_COL_CANDIDATES = ["fe_valor_ajuizado", "Valorajuizado", "valorajuizado", "ValorAjuizado"]

RUN_FAST = True  # deixe True para iteração rápida; False aumenta busca/tuning
USE_TEXT_FEATURES = True
ENABLE_PERMUTATION_IMPORTANCE = True

N_ITER_RF = 12 if RUN_FAST else 40
N_JOBS = -1

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports" / "modelagem_early_stage"
MODEL_DIR = PROJECT_ROOT / "models" / "modelagem_early_stage"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def save_report(df: pd.DataFrame, filename: str, index: bool = False):
    path = REPORT_DIR / filename
    df.to_csv(path, index=index, encoding="utf-8-sig")
    print(f"Relatório salvo em: {path}")
    return path

def display_full(df, n=20):
    with pd.option_context("display.max_columns", 200, "display.width", 240, "display.max_colwidth", 120):
        display(df.head(n))

print("REPORT_DIR:", REPORT_DIR)
print("MODEL_DIR:", MODEL_DIR)


In [ ]:
# ============================================================
# 2. Carregar bases dev/holdout e lista de features
# ============================================================

candidate_paths = {
    "dev_hist": DATA_DIR / "abt_early_stage_v1_dev_hist.parquet",
    "holdout_hist": DATA_DIR / "abt_early_stage_v1_holdout_hist.parquet",
    "features_hist": DATA_DIR / "feature_list_early_stage_v1_hist.txt",
    "dev_base": DATA_DIR / "abt_early_stage_v1_dev.parquet",
    "holdout_base": DATA_DIR / "abt_early_stage_v1_holdout.parquet",
    "features_base": DATA_DIR / "feature_list_early_stage_v1.txt",
}

if candidate_paths["dev_hist"].exists() and candidate_paths["holdout_hist"].exists():
    DEV_PATH = candidate_paths["dev_hist"]
    HOLDOUT_PATH = candidate_paths["holdout_hist"]
    FEATURE_LIST_PATH = candidate_paths["features_hist"] if candidate_paths["features_hist"].exists() else None
    DATA_VERSION = "hist"
else:
    DEV_PATH = candidate_paths["dev_base"]
    HOLDOUT_PATH = candidate_paths["holdout_base"]
    FEATURE_LIST_PATH = candidate_paths["features_base"] if candidate_paths["features_base"].exists() else None
    DATA_VERSION = "base"

print("DATA_VERSION:", DATA_VERSION)
print("DEV_PATH:", DEV_PATH)
print("HOLDOUT_PATH:", HOLDOUT_PATH)
print("FEATURE_LIST_PATH:", FEATURE_LIST_PATH)

if not DEV_PATH.exists():
    raise FileNotFoundError(f"Arquivo dev não encontrado: {DEV_PATH}")
if not HOLDOUT_PATH.exists():
    raise FileNotFoundError(f"Arquivo holdout não encontrado: {HOLDOUT_PATH}")

df_dev = pd.read_parquet(DEV_PATH)
df_holdout = pd.read_parquet(HOLDOUT_PATH)

if TARGET_COL not in df_dev.columns:
    raise ValueError(f"TARGET_COL não encontrado no dev: {TARGET_COL}")
if TARGET_COL not in df_holdout.columns:
    raise ValueError(f"TARGET_COL não encontrado no holdout: {TARGET_COL}")

df_dev[DATE_COL] = pd.to_datetime(df_dev[DATE_COL], errors="coerce")
df_holdout[DATE_COL] = pd.to_datetime(df_holdout[DATE_COL], errors="coerce")

if FEATURE_LIST_PATH is not None and FEATURE_LIST_PATH.exists():
    feature_list = [x.strip() for x in FEATURE_LIST_PATH.read_text(encoding="utf-8").splitlines() if x.strip()]
else:
    print("Lista de features não encontrada. Criando lista automática com colunas não bloqueadas.")
    feature_list = []

print("df_dev:", df_dev.shape)
print("df_holdout:", df_holdout.shape)
print("features lidas:", len(feature_list))


In [ ]:
# ============================================================
# 3. Auditoria defensiva de leakage e preparação da feature list
# ============================================================

ID_COLS = ["Numerodistribuicao", "Identificador", "numero_processo", "processo", "CPF", "Cpf_Autor"]

LEAKAGE_PATTERNS = [
    r"motivo.*encerr", r"encerr", r"desfecho", r"resultado", r"sentenc", r"condena",
    r"acord", r"dataencerr", r"baix[aa]", r"transito", r"procedente", r"improcedente",
    r"prob$", r"_prob$", r"provavel", r"dias_ate", r"dias_faltantes",
    r"status_texto", r"resultado_final", r"sentenca_tipo", r"valor_do_acordo",
    r"condenacao_valor", r"dano_moral", r"fase_processual",
]

ALLOWED_PATTERNS_EXCEPTIONS = [r"^fe_.*hist_", r"^fe_hist_"]

def is_allowed_exception(col: str) -> bool:
    c = str(col).lower()
    return any(re.search(p, c) for p in ALLOWED_PATTERNS_EXCEPTIONS)

def is_name_leakage(col: str) -> bool:
    c = str(col).lower()
    if is_allowed_exception(c):
        return False
    return any(re.search(p, c) for p in LEAKAGE_PATTERNS)

blocked_base = set([TARGET_COL] + ID_COLS)
available_features = [c for c in feature_list if c in df_dev.columns and c in df_holdout.columns]

if not available_features:
    available_features = [
        c for c in df_dev.columns 
        if c in df_holdout.columns 
        and c not in blocked_base
        and not is_name_leakage(c)
    ]

feature_audit = pd.DataFrame({
    "feature": available_features,
    "exists_dev": [c in df_dev.columns for c in available_features],
    "exists_holdout": [c in df_holdout.columns for c in available_features],
    "possible_leakage_by_name": [is_name_leakage(c) for c in available_features],
    "dtype_dev": [str(df_dev[c].dtype) if c in df_dev.columns else None for c in available_features],
    "missing_rate_dev": [df_dev[c].isna().mean() if c in df_dev.columns else np.nan for c in available_features],
    "missing_rate_holdout": [df_holdout[c].isna().mean() if c in df_holdout.columns else np.nan for c in available_features],
    "nunique_dev": [df_dev[c].nunique(dropna=True) if c in df_dev.columns else np.nan for c in available_features],
    "nunique_holdout": [df_holdout[c].nunique(dropna=True) if c in df_holdout.columns else np.nan for c in available_features],
})

save_report(feature_audit, "90_04_feature_inventory.csv")
display_full(feature_audit, 30)

leakage_found = feature_audit[feature_audit["possible_leakage_by_name"]]
if len(leakage_found) > 0:
    print("ATENÇÃO: removendo features suspeitas de leakage por nome.")
    display_full(leakage_found, 50)

feature_list_clean = [
    c for c in available_features
    if c in df_dev.columns
    and c in df_holdout.columns
    and not is_name_leakage(c)
    and c != TARGET_COL
]

print("Features após auditoria:", len(feature_list_clean))


In [ ]:
# ============================================================
# 4. Criar variável de texto combinado, detectar tipos e blocos
# ============================================================

TEXT_COL_REGEX = re.compile(r"(texto|descricao|descrição|assunto_text|classe_text|area_text|tipo_justica|ementa|pedido|resumo)", re.IGNORECASE)

raw_text_cols = [
    c for c in feature_list_clean
    if c in df_dev.columns
    and (
        TEXT_COL_REGEX.search(c)
        or (df_dev[c].dtype == "object" and df_dev[c].astype(str).str.len().median() > 40)
    )
]

TEXT_COMBINED_COL = "fe_texto_combinado_early"

def combine_text_cols(df: pd.DataFrame, text_cols):
    if not text_cols:
        return pd.Series([""] * len(df), index=df.index)
    return (
        df[text_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

if USE_TEXT_FEATURES and raw_text_cols:
    df_dev[TEXT_COMBINED_COL] = combine_text_cols(df_dev, raw_text_cols)
    df_holdout[TEXT_COMBINED_COL] = combine_text_cols(df_holdout, raw_text_cols)
    feature_list_model = [c for c in feature_list_clean if c not in raw_text_cols] + [TEXT_COMBINED_COL]
    text_features = [TEXT_COMBINED_COL]
else:
    feature_list_model = [c for c in feature_list_clean if c not in raw_text_cols]
    text_features = []

numeric_features = [c for c in feature_list_model if c in df_dev.columns and c not in text_features and pd.api.types.is_numeric_dtype(df_dev[c])]
categorical_features = [c for c in feature_list_model if c in df_dev.columns and c not in text_features and c not in numeric_features]

print("Features model:", len(feature_list_model))
print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))
print("Text raw:", len(raw_text_cols))
print("Text combined:", text_features)

def infer_block(feature: str) -> str:
    f = str(feature).lower()
    if feature == TEXT_COMBINED_COL or "tfidf" in f or TEXT_COL_REGEX.search(feature):
        return "texto"
    if "valor" in f or "ajuizado" in f or "faixa_valor" in f:
        return "valor"
    if "hist_" in f or f.startswith("fe_hist") or "rate_smooth" in f or "across_groups" in f:
        return "historicas_retrospectivas"
    if "inter_" in f:
        return "interacoes"
    if any(x in f for x in ["ano", "mes", "trimestre", "semestre", "cohort", "data", "dia_semana"]):
        return "temporal"
    if any(x in f for x in ["uf", "comarca", "vara"]):
        return "geografico_judicial"
    if any(x in f for x in ["assunto", "produto", "acao", "carteira", "empresa", "tipo", "classe"]):
        return "categorias_cadastro"
    return "outros"

feature_blocks = pd.DataFrame({
    "feature": feature_list_model,
    "block": [infer_block(c) for c in feature_list_model],
    "dtype": [str(df_dev[c].dtype) for c in feature_list_model],
})
save_report(feature_blocks, "90_04_feature_blocks.csv")
display(feature_blocks["block"].value_counts().rename("qtd_features"))


In [ ]:
# ============================================================
# 5. Criar folds walk-forward dentro do Dev
# ============================================================

def make_temporal_folds(df, date_col=DATE_COL, n_folds=3):
    tmp = df[[date_col, TARGET_COL]].copy().sort_values(date_col)
    boundaries = [0.55, 0.70, 0.85, 1.00]
    boundaries = boundaries[-(n_folds+1):]
    folds, rows = [], []
    n = len(tmp)
    sorted_idx = tmp.index.to_numpy()
    for i in range(n_folds):
        train_end_pos = int(boundaries[i] * n)
        val_end_pos = int(boundaries[i+1] * n)
        train_idx = sorted_idx[:train_end_pos]
        val_idx = sorted_idx[train_end_pos:val_end_pos]
        if len(train_idx) == 0 or len(val_idx) == 0:
            continue
        y_tr = df.loc[train_idx, TARGET_COL]
        y_va = df.loc[val_idx, TARGET_COL]
        rows.append({
            "fold": i + 1,
            "train_start_date": df.loc[train_idx, date_col].min(),
            "train_end_date": df.loc[train_idx, date_col].max(),
            "val_start_date": df.loc[val_idx, date_col].min(),
            "val_end_date": df.loc[val_idx, date_col].max(),
            "qtd_train": len(train_idx),
            "qtd_val": len(val_idx),
            "taxa_perda_train": y_tr.mean(),
            "taxa_perda_val": y_va.mean(),
            "taxa_ganho_train": 1 - y_tr.mean(),
            "taxa_ganho_val": 1 - y_va.mean(),
        })
        folds.append((train_idx, val_idx))
    return folds, pd.DataFrame(rows)

df_dev = df_dev.sort_values(DATE_COL).reset_index(drop=True)
df_holdout = df_holdout.sort_values(DATE_COL).reset_index(drop=True)
folds, fold_summary = make_temporal_folds(df_dev, DATE_COL, n_folds=3)
save_report(fold_summary, "91_04_walk_forward_folds_summary.csv")
display(fold_summary)


In [ ]:
# ============================================================
# 6. Métricas e helpers
# ============================================================

def get_proba_pos(estimator, X):
    if hasattr(estimator, "predict_proba"):
        proba = estimator.predict_proba(X)
        if proba.ndim == 2:
            if hasattr(estimator, "classes_"):
                classes = list(estimator.classes_)
                if POS_LABEL in classes:
                    return proba[:, classes.index(POS_LABEL)]
            return proba[:, 1]
        return proba
    if hasattr(estimator, "decision_function"):
        scores = estimator.decision_function(X)
        return 1 / (1 + np.exp(-scores))
    raise ValueError("Estimator não possui predict_proba nem decision_function.")

def best_f1_threshold(y_true, proba):
    precision, recall, thresholds = precision_recall_curve(y_true, proba, pos_label=POS_LABEL)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    if len(thresholds) == 0:
        return 0.5, np.nan, np.nan, np.nan
    best_idx = int(np.nanargmax(f1[:-1]))
    return float(thresholds[best_idx]), float(precision[best_idx]), float(recall[best_idx]), float(f1[best_idx])

def metrics_from_proba(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    best_thr, best_p, best_r, best_f1 = best_f1_threshold(y_true, proba)
    return {
        "roc_auc_perda": roc_auc_score(y_true, proba),
        "pr_auc_perda": average_precision_score(y_true, proba, pos_label=POS_LABEL),
        "threshold": threshold,
        "precision_perda_t05": precision_score(y_true, pred, pos_label=POS_LABEL, zero_division=0),
        "recall_perda_t05": recall_score(y_true, pred, pos_label=POS_LABEL, zero_division=0),
        "f1_perda_t05": f1_score(y_true, pred, pos_label=POS_LABEL, zero_division=0),
        "pred_perda_rate_t05": float(pred.mean()),
        "best_f1_threshold_perda": best_thr,
        "best_f1_precision_perda": best_p,
        "best_f1_recall_perda": best_r,
        "best_f1_perda": best_f1,
        "taxa_perda": float(np.mean(y_true)),
        "taxa_ganho": float(1 - np.mean(y_true)),
    }

def topk_risco_perda_metrics(y_true, proba_perda, ks=(0.01, 0.05, 0.10, 0.20)):
    df = pd.DataFrame({"y": np.asarray(y_true), "proba_perda": np.asarray(proba_perda)})
    df = df.sort_values("proba_perda", ascending=False).reset_index(drop=True)
    base_rate = df["y"].mean()
    rows = []
    for k in ks:
        n_top = max(1, int(round(len(df) * k)))
        top = df.head(n_top)
        precision_at_k = top["y"].mean()
        recall_at_k = top["y"].sum() / max(df["y"].sum(), 1)
        rows.append({
            "top_k": k,
            "n_top": n_top,
            "precision_perda_at_k": precision_at_k,
            "recall_perda_at_k": recall_at_k,
            "lift_perda_at_k": precision_at_k / base_rate if base_rate > 0 else np.nan,
            "taxa_perda_base": base_rate,
        })
    return pd.DataFrame(rows)

def topk_prioridade_financeira_metrics(y_true, proba_perda, valor_ajuizado, ks=(0.01, 0.05, 0.10, 0.20)):
    df = pd.DataFrame({
        "y": np.asarray(y_true),
        "proba_perda": np.asarray(proba_perda),
        "valor_ajuizado": pd.to_numeric(valor_ajuizado, errors="coerce").fillna(0).values,
    })
    df["perda_real_valor"] = df["y"] * df["valor_ajuizado"]
    df["prioridade_financeira"] = df["proba_perda"] * df["valor_ajuizado"]
    df = df.sort_values("prioridade_financeira", ascending=False).reset_index(drop=True)
    base_rate = df["y"].mean()
    total_valor = df["valor_ajuizado"].sum()
    total_perdas = df["perda_real_valor"].sum()
    rows = []
    for k in ks:
        n_top = max(1, int(round(len(df) * k)))
        top = df.head(n_top)
        precision_at_k = top["y"].mean()
        rows.append({
            "top_k": k,
            "n_top": n_top,
            "precision_perda_at_k": precision_at_k,
            "recall_perda_at_k": top["y"].sum() / max(df["y"].sum(), 1),
            "lift_perda_at_k": precision_at_k / base_rate if base_rate > 0 else np.nan,
            "taxa_perda_base": base_rate,
            "valor_ajuizado_top": top["valor_ajuizado"].sum(),
            "share_valor_ajuizado_total": top["valor_ajuizado"].sum() / total_valor if total_valor > 0 else np.nan,
            "valor_ajuizado_perdas_top": top["perda_real_valor"].sum(),
            "share_valor_perdas_total": top["perda_real_valor"].sum() / total_perdas if total_perdas > 0 else np.nan,
        })
    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# 7. Preprocessador sklearn
# ============================================================

def make_preprocessor(numeric_features, categorical_features, text_features,
                      tfidf_max_features=1500, tfidf_min_df=20, onehot_min_frequency=20,
                      scale_numeric=False):
    transformers = []
    if numeric_features:
        num_steps = [("imputer", SimpleImputer(strategy="median"))]
        if scale_numeric:
            num_steps.append(("scaler", StandardScaler()))
        transformers.append(("num", Pipeline(num_steps), numeric_features))
    if categorical_features:
        categorical_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=onehot_min_frequency, sparse_output=True)),
        ])
        transformers.append(("cat", categorical_pipeline, categorical_features))
    if text_features:
        text_col = text_features[0]
        text_pipeline = Pipeline([
            ("fillna", FunctionTransformer(lambda x: pd.Series(x.squeeze()).fillna("").astype(str), validate=False)),
            ("tfidf", TfidfVectorizer(max_features=tfidf_max_features, min_df=tfidf_min_df, ngram_range=(1, 2), strip_accents="unicode", lowercase=True)),
        ])
        transformers.append(("txt", text_pipeline, text_col))
    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.3, verbose_feature_names_out=True)

def make_rf_pipeline(numeric_features, categorical_features, text_features, rf_params=None,
                     tfidf_max_features=1500, tfidf_min_df=20, onehot_min_frequency=20):
    rf_params = rf_params or {}
    preprocessor = make_preprocessor(numeric_features, categorical_features, text_features, tfidf_max_features, tfidf_min_df, onehot_min_frequency, scale_numeric=False)
    model = RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_split=20, min_samples_leaf=10,
        max_features="sqrt", class_weight="balanced", bootstrap=True, n_jobs=N_JOBS,
        random_state=RANDOM_STATE, **rf_params,
    )
    return Pipeline([("preprocess", preprocessor), ("model", model)])

def make_lr_pipeline(numeric_features, categorical_features, text_features,
                     tfidf_max_features=1500, tfidf_min_df=20, onehot_min_frequency=20):
    preprocessor = make_preprocessor(numeric_features, categorical_features, text_features, tfidf_max_features, tfidf_min_df, onehot_min_frequency, scale_numeric=True)
    model = LogisticRegression(C=1.0, penalty="l2", solver="saga", max_iter=1000, class_weight="balanced", n_jobs=N_JOBS, random_state=RANDOM_STATE)
    return Pipeline([("preprocess", preprocessor), ("model", model)])


In [ ]:
# ============================================================
# 8. Avaliação walk-forward para um pipeline
# ============================================================

def evaluate_pipeline_walk_forward(name, pipeline, df, features, folds):
    rows, topk_rows = [], []
    for fold_id, (train_idx, val_idx) in enumerate(folds, start=1):
        X_train = df.loc[train_idx, features]
        y_train = df.loc[train_idx, TARGET_COL].astype(int)
        X_val = df.loc[val_idx, features]
        y_val = df.loc[val_idx, TARGET_COL].astype(int)
        pipeline.fit(X_train, y_train)
        proba = get_proba_pos(pipeline, X_val)
        met = metrics_from_proba(y_val, proba, threshold=0.5)
        met.update({
            "model": name, "fold": fold_id, "qtd_train": len(train_idx), "qtd_val": len(val_idx),
            "train_start_date": df.loc[train_idx, DATE_COL].min(),
            "train_end_date": df.loc[train_idx, DATE_COL].max(),
            "val_start_date": df.loc[val_idx, DATE_COL].min(),
            "val_end_date": df.loc[val_idx, DATE_COL].max(),
        })
        rows.append(met)
        tk = topk_risco_perda_metrics(y_val, proba)
        tk["model"] = name
        tk["fold"] = fold_id
        topk_rows.append(tk)
    return pd.DataFrame(rows), pd.concat(topk_rows, ignore_index=True)


In [ ]:
# ============================================================
# 9. Ablation test por blocos de features
# ============================================================

block_map = feature_blocks.set_index("feature")["block"].to_dict()

def select_features_without_blocks(features, blocks_to_remove):
    return [f for f in features if block_map.get(f, "outros") not in set(blocks_to_remove)]

ablation_specs = [
    ("full", []),
    ("sem_texto", ["texto"]),
    ("sem_historicas_retrospectivas", ["historicas_retrospectivas"]),
    ("sem_interacoes", ["interacoes"]),
    ("sem_valor", ["valor"]),
    ("sem_categorias_cadastro", ["categorias_cadastro"]),
    ("sem_geografico_judicial", ["geografico_judicial"]),
    ("sem_temporal", ["temporal"]),
]

ablation_results, ablation_topk_results = [], []
for spec_name, blocks_to_remove in ablation_specs:
    feats = select_features_without_blocks(feature_list_model, blocks_to_remove)
    if len(feats) == 0:
        continue
    num_f = [f for f in feats if f in numeric_features]
    cat_f = [f for f in feats if f in categorical_features]
    txt_f = [f for f in feats if f in text_features] if USE_TEXT_FEATURES else []
    print(f"\nAblation: {spec_name} | features={len(feats)} | num={len(num_f)} cat={len(cat_f)} text={len(txt_f)}")
    pipe = make_rf_pipeline(num_f, cat_f, txt_f)
    fold_metrics, fold_topk = evaluate_pipeline_walk_forward(f"rf_{spec_name}", pipe, df_dev, feats, folds)
    fold_metrics["spec"] = spec_name
    fold_metrics["removed_blocks"] = ",".join(blocks_to_remove) if blocks_to_remove else ""
    fold_metrics["qtd_features"] = len(feats)
    ablation_results.append(fold_metrics)
    fold_topk["spec"] = spec_name
    fold_topk["qtd_features"] = len(feats)
    ablation_topk_results.append(fold_topk)

ablation_fold_metrics = pd.concat(ablation_results, ignore_index=True)
ablation_topk = pd.concat(ablation_topk_results, ignore_index=True)

ablation_summary = (
    ablation_fold_metrics
    .groupby(["spec", "removed_blocks", "qtd_features"], dropna=False)
    .agg(
        mean_roc_auc_perda=("roc_auc_perda", "mean"),
        std_roc_auc_perda=("roc_auc_perda", "std"),
        mean_pr_auc_perda=("pr_auc_perda", "mean"),
        std_pr_auc_perda=("pr_auc_perda", "std"),
        mean_best_f1_perda=("best_f1_perda", "mean"),
        mean_precision_perda_t05=("precision_perda_t05", "mean"),
        mean_recall_perda_t05=("recall_perda_t05", "mean"),
        mean_f1_perda_t05=("f1_perda_t05", "mean"),
    )
    .reset_index()
    .sort_values("mean_pr_auc_perda", ascending=False)
)

save_report(ablation_fold_metrics, "91_04_block_ablation_fold_metrics.csv")
save_report(ablation_summary, "91_04_block_ablation_summary.csv")
save_report(ablation_topk, "91_04_block_ablation_topk.csv")
display(ablation_summary)


In [ ]:
# ============================================================
# 10. Seleção não supervisionada por correlação em numéricas
#     Observação: não usa target, portanto não vaza target.
# ============================================================

def correlation_filter_numeric(df, numeric_features, threshold=0.98):
    if len(numeric_features) <= 1:
        return numeric_features, []
    X = df[numeric_features].copy().replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))
    corr = X.corr(method="spearman").abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    selected = [c for c in numeric_features if c not in to_drop]
    return selected, to_drop

numeric_selected_corr, numeric_dropped_corr = correlation_filter_numeric(df_dev, numeric_features, threshold=0.98)

corr_selection_report = pd.DataFrame({
    "feature": numeric_features,
    "selected_after_corr_filter": [f in numeric_selected_corr for f in numeric_features],
    "dropped_corr_gt_098": [f in numeric_dropped_corr for f in numeric_features],
})

save_report(corr_selection_report, "92_04_numeric_corr_selection.csv")
print("Numeric antes:", len(numeric_features))
print("Numeric depois:", len(numeric_selected_corr))
print("Dropadas:", len(numeric_dropped_corr))
display_full(corr_selection_report[corr_selection_report["dropped_corr_gt_098"]], 50)


In [ ]:
# ============================================================
# 11. Definir feature set para tuning
# ============================================================

USE_CORR_FILTERED_NUMERIC = True

tuning_numeric_features = numeric_selected_corr if USE_CORR_FILTERED_NUMERIC else numeric_features
tuning_categorical_features = categorical_features
tuning_text_features = text_features if USE_TEXT_FEATURES else []
tuning_features = tuning_numeric_features + tuning_categorical_features + tuning_text_features

print("Tuning features:", len(tuning_features))
print("Numeric:", len(tuning_numeric_features))
print("Categorical:", len(tuning_categorical_features))
print("Text:", len(tuning_text_features))

missing_tuning = [c for c in tuning_features if c not in df_dev.columns or c not in df_holdout.columns]
if missing_tuning:
    raise ValueError(f"Features de tuning ausentes: {missing_tuning[:20]}")


In [ ]:
# ============================================================
# 12. Hyperparameter tuning — RandomForest campeão atual
# ============================================================

X_dev = df_dev[tuning_features].copy()
y_dev = df_dev[TARGET_COL].astype(int).copy()
cv_splits = [(train_idx, val_idx) for train_idx, val_idx in folds]

rf_pipe = make_rf_pipeline(tuning_numeric_features, tuning_categorical_features, tuning_text_features, tfidf_max_features=1500, tfidf_min_df=20, onehot_min_frequency=20)

param_distributions = {
    "model__n_estimators": randint(300, 900),
    "model__max_depth": [8, 12, 16, 20, 24, None],
    "model__min_samples_split": randint(10, 80),
    "model__min_samples_leaf": randint(5, 40),
    "model__max_features": ["sqrt", "log2", 0.25, 0.4, 0.6],
    "model__class_weight": ["balanced", "balanced_subsample", {0: 1.0, 1: 2.0}, {0: 1.0, 1: 3.0}],
    "model__criterion": ["gini", "entropy", "log_loss"],
    "model__max_samples": [None, 0.7, 0.8, 0.9],
}

try:
    ap_scorer = make_scorer(average_precision_score, response_method="predict_proba", pos_label=POS_LABEL)
except TypeError:
    ap_scorer = make_scorer(average_precision_score, needs_proba=True, pos_label=POS_LABEL)

search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=param_distributions,
    n_iter=N_ITER_RF,
    scoring=ap_scorer,
    cv=cv_splits,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE,
    verbose=2,
    refit=True,
    return_train_score=True,
)

search.fit(X_dev, y_dev)

print("Best score mean PR-AUC perda:", search.best_score_)
print("Best params:")
print(json.dumps(search.best_params_, indent=2, default=str))

tuning_results = pd.DataFrame(search.cv_results_).sort_values("mean_test_score", ascending=False)
save_report(tuning_results, "92_04_rf_tuning_results.csv")
display_full(tuning_results[["mean_test_score", "std_test_score", "mean_train_score", "params"]], 20)


In [ ]:
# ============================================================
# 13. Avaliar melhor modelo tunado no walk-forward e no holdout
# ============================================================

best_model = search.best_estimator_

tuned_fold_metrics, tuned_fold_topk = evaluate_pipeline_walk_forward(
    name="rf_tuned_07_04",
    pipeline=best_model,
    df=df_dev,
    features=tuning_features,
    folds=folds,
)

save_report(tuned_fold_metrics, "93_04_rf_tuned_walk_forward_fold_metrics.csv")
save_report(tuned_fold_topk, "93_04_rf_tuned_walk_forward_topk.csv")
display(tuned_fold_metrics)

X_holdout = df_holdout[tuning_features].copy()
y_holdout = df_holdout[TARGET_COL].astype(int).copy()

best_model.fit(X_dev, y_dev)
proba_perda_holdout = get_proba_pos(best_model, X_holdout)
holdout_m = metrics_from_proba(y_holdout, proba_perda_holdout, threshold=0.5)

holdout_metrics = pd.DataFrame([{
    "model": "rf_tuned_07_04",
    "data_version": DATA_VERSION,
    "target_semantica": "0=banco_ganhou | 1=banco_perdeu",
    "roc_auc_perda": holdout_m["roc_auc_perda"],
    "pr_auc_perda": holdout_m["pr_auc_perda"],
    "taxa_perda_holdout": holdout_m["taxa_perda"],
    "taxa_ganho_holdout": holdout_m["taxa_ganho"],
    "best_f1_threshold_perda": holdout_m["best_f1_threshold_perda"],
    "best_f1_precision_perda": holdout_m["best_f1_precision_perda"],
    "best_f1_recall_perda": holdout_m["best_f1_recall_perda"],
    "best_f1_perda": holdout_m["best_f1_perda"],
    "precision_perda_t05": holdout_m["precision_perda_t05"],
    "recall_perda_t05": holdout_m["recall_perda_t05"],
    "f1_perda_t05": holdout_m["f1_perda_t05"],
    "pred_perda_rate_t05": holdout_m["pred_perda_rate_t05"],
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "qtd_features": len(tuning_features),
    "use_text_features": USE_TEXT_FEATURES,
    "best_cv_pr_auc_perda": search.best_score_,
}])

save_report(holdout_metrics, "93_04_holdout_best_tuned_metrics.csv")
display(holdout_metrics.T)

print("\nClassification report — threshold 0.5")
print("Classe 0 = banco ganhou")
print("Classe 1 = banco perdeu")
pred_t05 = (proba_perda_holdout >= 0.5).astype(int)
print(classification_report(y_holdout, pred_t05, target_names=["banco_ganhou", "banco_perdeu"], zero_division=0))
print("Confusion matrix — linhas=real | colunas=previsto")
print(confusion_matrix(y_holdout, pred_t05))


In [ ]:
# ============================================================
# 14. Top-k risco de perda e prioridade financeira
# ============================================================

holdout_topk_risco = topk_risco_perda_metrics(y_true=y_holdout.values, proba_perda=proba_perda_holdout, ks=(0.01, 0.05, 0.10, 0.20))
holdout_topk_risco["model"] = "rf_tuned_07_04"
save_report(holdout_topk_risco, "94_04_holdout_topk_risco_perda.csv")
display(holdout_topk_risco)

VALUE_COL = next((c for c in VALUE_COL_CANDIDATES if c in df_holdout.columns), None)

if VALUE_COL:
    print("VALUE_COL:", VALUE_COL)
    holdout_topk_fin = topk_prioridade_financeira_metrics(
        y_true=y_holdout.values,
        proba_perda=proba_perda_holdout,
        valor_ajuizado=df_holdout[VALUE_COL],
        ks=(0.01, 0.05, 0.10, 0.20),
    )
    holdout_topk_fin["model"] = "rf_tuned_07_04"
    holdout_topk_fin["value_col"] = VALUE_COL
    save_report(holdout_topk_fin, "95_04_holdout_topk_prioridade_financeira.csv")
    display(holdout_topk_fin)
else:
    print("Nenhuma coluna de valor encontrada para prioridade financeira.")
    holdout_topk_fin = pd.DataFrame()


In [ ]:
# ============================================================
# 15. Ranking do holdout para consumo jurídico
# ============================================================

ranking_cols = []
for c in ["Numerodistribuicao", "Identificador", DATE_COL, VALUE_COL, "Nm_Assunto", "Nm_Acao", "Nm_Produto", "Carteira", "UF", "Comarca"]:
    if c and c in df_holdout.columns and c not in ranking_cols:
        ranking_cols.append(c)

ranking_holdout = df_holdout[ranking_cols].copy() if ranking_cols else pd.DataFrame(index=df_holdout.index)
ranking_holdout[TARGET_COL] = y_holdout.values
ranking_holdout["target_label"] = np.where(ranking_holdout[TARGET_COL] == 1, "banco_perdeu", "banco_ganhou")
ranking_holdout["proba_perda_raw"] = proba_perda_holdout
ranking_holdout["score_risco_perda"] = proba_perda_holdout
if VALUE_COL:
    ranking_holdout["score_prioridade_financeira"] = proba_perda_holdout * pd.to_numeric(df_holdout[VALUE_COL], errors="coerce").fillna(0).values
else:
    ranking_holdout["score_prioridade_financeira"] = np.nan

ranking_holdout["rank_risco_perda"] = ranking_holdout["score_risco_perda"].rank(method="first", ascending=False).astype(int)
ranking_holdout["rank_prioridade_financeira"] = ranking_holdout["score_prioridade_financeira"].rank(method="first", ascending=False).astype(int)
ranking_holdout = ranking_holdout.sort_values("rank_risco_perda")

ranking_path = DATA_DIR / "ranking_holdout_risco_perda_07_04.parquet"
ranking_holdout.to_parquet(ranking_path, index=False)
print("Ranking holdout salvo em:", ranking_path)

save_report(ranking_holdout.head(1000), "96_04_ranking_holdout_top1000_risco_perda.csv")
display_full(ranking_holdout, 20)


In [ ]:
# ============================================================
# 16. Permutation importance no holdout
#     Usar apenas para interpretação/diagnóstico, não para retreinar automaticamente.
# ============================================================

if ENABLE_PERMUTATION_IMPORTANCE:
    try:
        perm = permutation_importance(
            best_model,
            X_holdout,
            y_holdout,
            scoring=ap_scorer,
            n_repeats=5 if RUN_FAST else 10,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )
        perm_df = pd.DataFrame({
            "feature": tuning_features,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
        }).sort_values("importance_mean", ascending=False)
        save_report(perm_df, "96_04_permutation_importance_holdout.csv")
        display_full(perm_df, 50)
    except Exception as e:
        print("Permutation importance falhou:", repr(e))
        perm_df = pd.DataFrame()
else:
    perm_df = pd.DataFrame()
    print("Permutation importance desabilitado.")


In [ ]:
# ============================================================
# 17. Salvar modelo final e lista de features
# ============================================================

model_path = MODEL_DIR / "best_model_07_04.joblib"
joblib.dump(best_model, model_path)

feature_path = DATA_DIR / "feature_list_early_stage_v1_tuned_07_04.txt"
feature_path.write_text("\n".join(tuning_features), encoding="utf-8")

metadata = {
    "created_at": datetime.now().isoformat(),
    "notebook": "07_04_feature_selection_hyperparameter_tuning.ipynb",
    "target_col": TARGET_COL,
    "target_semantica": "0=banco_ganhou | 1=banco_perdeu",
    "pos_label": POS_LABEL,
    "date_col": DATE_COL,
    "data_version": DATA_VERSION,
    "dev_path": str(DEV_PATH),
    "holdout_path": str(HOLDOUT_PATH),
    "feature_list_path": str(feature_path),
    "model_path": str(model_path),
    "qtd_features": len(tuning_features),
    "best_params": search.best_params_,
    "best_cv_pr_auc_perda": float(search.best_score_),
    "holdout_pr_auc_perda": float(holdout_m["pr_auc_perda"]),
    "holdout_roc_auc_perda": float(holdout_m["roc_auc_perda"]),
}

metadata_path = MODEL_DIR / "best_model_07_04_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")

print("Modelo salvo em:", model_path)
print("Features salvas em:", feature_path)
print("Metadata salva em:", metadata_path)


In [ ]:
# ============================================================
# 18. Comparação com baselines anteriores, se existirem
# ============================================================

comparison_rows = []
comparison_rows.append({"model": "07_03A_rf_onehot_tfidf_referencia", "holdout_pr_auc_perda": 0.466804, "holdout_roc_auc_perda": 0.778580, "top5_precision_perda": 0.603004, "source": "referencia_conversa"})
comparison_rows.append({"model": "07_03D_rf_onehot_tfidf_texto_referencia", "holdout_pr_auc_perda": 0.470186, "holdout_roc_auc_perda": 0.778536, "top5_precision_perda": 0.613734, "source": "referencia_conversa"})

top5_row = holdout_topk_risco.loc[holdout_topk_risco["top_k"] == 0.05].iloc[0].to_dict()
comparison_rows.append({"model": "07_04_rf_tuned", "holdout_pr_auc_perda": holdout_m["pr_auc_perda"], "holdout_roc_auc_perda": holdout_m["roc_auc_perda"], "top5_precision_perda": top5_row["precision_perda_at_k"], "source": "notebook_07_04"})

comparison = pd.DataFrame(comparison_rows)
base_pr = comparison.loc[comparison["model"] == "07_03D_rf_onehot_tfidf_texto_referencia", "holdout_pr_auc_perda"].iloc[0]
base_top5 = comparison.loc[comparison["model"] == "07_03D_rf_onehot_tfidf_texto_referencia", "top5_precision_perda"].iloc[0]
comparison["delta_pr_auc_vs_07_03D"] = comparison["holdout_pr_auc_perda"] - base_pr
comparison["delta_top5_precision_vs_07_03D"] = comparison["top5_precision_perda"] - base_top5

save_report(comparison, "97_04_comparison_vs_previous_baselines.csv")
display(comparison)


In [ ]:
# ============================================================
# 19. Summary final
# ============================================================

summary_step_04 = pd.DataFrame([{
    "target_semantica": "0=banco_ganhou | 1=banco_perdeu",
    "data_version": DATA_VERSION,
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "taxa_perda_dev": float(y_dev.mean()),
    "taxa_perda_holdout": float(y_holdout.mean()),
    "qtd_features_inicio": len(feature_list_model),
    "qtd_features_tuning": len(tuning_features),
    "qtd_numeric_tuning": len(tuning_numeric_features),
    "qtd_categorical_tuning": len(tuning_categorical_features),
    "qtd_text_tuning": len(tuning_text_features),
    "best_cv_pr_auc_perda": float(search.best_score_),
    "holdout_pr_auc_perda": float(holdout_m["pr_auc_perda"]),
    "holdout_roc_auc_perda": float(holdout_m["roc_auc_perda"]),
    "holdout_best_f1_threshold_perda": float(holdout_m["best_f1_threshold_perda"]),
    "holdout_best_f1_perda": float(holdout_m["best_f1_perda"]),
    "holdout_precision_perda_t05": float(holdout_m["precision_perda_t05"]),
    "holdout_recall_perda_t05": float(holdout_m["recall_perda_t05"]),
    "ranking_holdout_path": str(ranking_path),
    "model_path": str(model_path),
    "feature_path": str(feature_path),
}])

save_report(summary_step_04, "98_04_summary_step_04.csv")
display(summary_step_04.T)

print("""
O que enviar na próxima iteração:

1. summary_step_04
2. ablation_summary
3. tuning_results.head(10)[["mean_test_score", "std_test_score", "params"]]
4. holdout_metrics.T
5. holdout_topk_risco
6. holdout_topk_fin, se VALUE_COL existir
7. comparison
8. permutation importance top 30, se rodou
""")


## Leitura esperada dos resultados

Como o target é `1 = banco perdeu`, as métricas com sufixo `_perda` medem a capacidade de identificar processos com maior risco de perda para o banco.

### Como interpretar

- `PR-AUC perda`: principal métrica de ranking em base desbalanceada.
- `ROC-AUC perda`: separabilidade geral entre perda e ganho.
- `precision_perda_at_k`: entre os top-k% mais arriscados, qual percentual realmente foi perda.
- `lift_perda_at_k`: quanto melhor o ranking é contra a taxa base de perda.
- `share_valor_perdas_total`: quanto do valor ajuizado perdido foi capturado no top-k financeiro.

### Próxima etapa recomendada

Após validar o melhor modelo tunado:

```text
07_05_calibracao_threshold_conformal.ipynb
```

Esse próximo notebook deve separar:

```text
score_risco_perda bruto -> ranking
probabilidade calibrada -> comunicação e faixas de risco
```

E testar:

```text
sigmoid / Platt
isotonic
Venn-Abers, se permitido no ambiente
threshold operacional
top-k operacional
```
